# Building with Mistral Models 

## Introduction 

This lesson will cover: 
- Exploring the different Mistral Models 
- Understanding the use-cases and scenarios for each model 
- Code samples show the unique features of each model. 

## The Mistral Models 

In this lesson, we will explore 3 different Mistral models: 
**Mistral Large**, **Mistral Small** and **Mistral Nemo**. 

Each of these models are available free on the Github Model marketplace. The code in this notebook will be using this models to run the code. Here are more details on using Github Models to [prototype with AI models](https://docs.github.com/en/github-models/prototyping-with-ai-models?WT.mc_id=academic-105485-koreyst). 


## Mistral Large 2 (2407)
Mistral Large 2 is currently the flagship model from Mistral and is designed for enterprise use. 

The model is an  upgrade to the original Mistral Large by offering 
-  Larger Context Window - 128k vs 32k 
-  Better performance on Math and Coding Tasks - 76.9% average accuracy vs 60.4% 
-  Increased multilingual performance - languages include: English, French, German, Spanish, Italian, Portuguese, Dutch, Russian, Chinese, Japanese, Korean, Arabic, and Hindi.

With these features, Mistral Large excels at 
- *Retrieval Augmented Generation (RAG)* - due to the larger context window
- *Function Calling* - this model has native function calling which allows integration with external tools and APIs. These calls can be made both in parallel or one after another in a sequential order. 
- *Code Generation* - this model excels on Python, Java, TypeScript and C++ generation. 

### RAG Example using Mistral Large 2 

In this example, we are using Mistral Large 2 to run a RAG pattern over a text document. The question is written in Korean and asks about the author's activities before college. 

It uses Cohere Embeddings Model to create embeddings of the text document as well as the question. For this sample, it uses the faiss Python package as a vector store. 

The prompt sent to the Mistral model includes both the questions and the retrieved chunks that are similar to the question. The Model then provides a natural language response. 

In [ ]:
from mistralai import Mistral  # 👉 đúng import
import os
import requests
import numpy as np
import faiss

# 1. Khởi tạo client
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key) 

# 2. Chuẩn bị dữ liệu & embeddings
response = requests.get("https://raw.githubusercontent.com/.../paul_graham_essay.txt")
text = response.text
chunks = [text[i:i+2048] for i in range(0, len(text), 2048)]

embed_response = client.embeddings.create(
    model="mistral-embed",
    inputs=chunks
)

text_embeddings = np.array([item.embedding for item in embed_response.data])
d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)

# 3. Tạo prompt từ truy vấn
question = "저자가 대학에 오기 전에 주로 했던 두 가지 일은 무엇이었나요?"
q_emb = client.embeddings.create(model="mistral-embed", inputs=[question])
q_vec = np.array(q_emb.data[0].embedding)

D, I = index.search(q_vec.reshape(1, -1), k=2)
retrieved_chunks = [chunks[i] for i in I[0]]

prompt = f"""
Context:
{retrieved_chunks}
Question: {question}
Answer in Vietnamese:
"""

# 4. Gọi chat completion
chat_response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        {"role": "system", "content": "You answer questions in Vietnamese."},#based on the context
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
    max_tokens=1000,
)

print(chat_response.choices[0].message.content)

Xin lỗi, nhưng tôi chỉ có thể trả lời bằng tiếng Việt hoặc tiếng Anh. Tuy nhiên, tôi có thể giúp bạn dịch câu trả lời sang tiếng Hàn sau khi trả lời bằng tiếng Việt.

Đáp án cho câu hỏi của bạn là: Hai việc chính mà tác giả thường làm trước khi vào đại học là:
1. Đọc sách và tìm hiểu kiến thức.
2. Viết bài và tham gia các hoạt động văn học để cải thiện kỹ năng viết.

B â n muốn tôi dịch câu trả lời này sang tiếng Hàn?


## Mistral Small 
Mistral Small is another model in the Mistral family of models under the premier/enterprise category. As the name implies, this model is a Small Language Model (SLM). The advantages of using Mistral Small are that it is: 
- Cost Saving compared to Mistral LLMs like Mistral Large and NeMo - 80% price drop
- Low latency - faster response compared to Mistral's LLMs
- Flexible - can be deployed across different environments with less restrictions on required resources. 


Mistral Small is great for: 
- Text based tasks such as summarization, sentiment analysis and translation. 
- Applications where frequent requests are made due to its cost effectiveness 
- Low latency code tasks like review and code suggestions 


## Comparing Mistral Small and Mistral Large 

To show differences in latency between Mistral Small and Large, run the below cells. 

You should see a difference in response times between 3-5 seconds. Also not the response lengths and style over the smae prompt.  

In [2]:
import os
from mistralai import Mistral
from mistralai.models import SystemMessage, UserMessage

# Lấy API key từ biến môi trường (GITHUB_TOKEN hoặc MISTRAL_API_KEY)
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

# Chuẩn bị prompt
messages = [
    SystemMessage(content="You are a helpful coding assistant."),
    UserMessage(content="Can you write a Python function to do the fizz buzz test?")
]

# Gọi chat completion
response = client.chat.complete(
    model="mistral-small-latest",  # hoặc mistral-large-latest
    messages=messages,
    temperature=1.0,
    top_p=1.0,
    max_tokens=500
)

# In kết quả trả về
print(response.choices[0].message.content)


Sure! The FizzBuzz test is a common programming problem where you print numbers from 1 to a given number, but for multiples of 3, you print "Fizz" instead of the number, and for multiples of 5, you print "Buzz". For numbers that are multiples of both 3 and 5, you print "FizzBuzz".

Here's a Python function to achieve that:

```python
def fizz_buzz(n):
    for i in range(1, n + 1):
        if i % 3 == 0 and i % 5 == 0:
            print("FizzBuzz")
        elif i % 3 == 0:
            print("Fizz")
        elif i % 5 == 0:
            print("Buzz")
        else:
            print(i)

# Example usage:
fizz_buzz(15)
```

This function takes an integer `n` as input and prints the FizzBuzz sequence up to that number. You can call `fizz_buzz(15)` to see the output for numbers from 1 to 15.


In [3]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Lấy API key từ biến môi trường (GITHUB_TOKEN hoặc MISTRAL_API_KEY)
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)
messages=[
        SystemMessage(content="You are a helpful coding assistant."),
        UserMessage(content="Can you write a Python function to the fizz buzz test?"),
    ]

response = client.chat.complete(
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model="mistral-large-latest",
    messages=messages
)

print(response.choices[0].message.content)

Certainly! The FizzBuzz test is a common programming problem where you are asked to print the numbers from 1 to a given number, but for multiples of three, print "Fizz" instead of the number and for the multiples of five print "Buzz". For numbers which are multiples of both three and five print "FizzBuzz".

Here's a Python function to accomplish this:

```python
def fizz_buzz(n):
    for i in range(1, n + 1):
        if i % 15 == 0:
            print("FizzBuzz")
        elif i % 3 == 0:
            print("Fizz")
        elif i % 5 == 0:
            print("Buzz")
        else:
            print(i)

# Example usage:
fizz_buzz(15)
```

In this function:
- We loop through the numbers from 1 to `n`.
- For each number, we check if it is divisible by 15 (which means it is divisible by both 3 and 5) and print "FizzBuzz".
- If it is not divisible by 15, we check if it is divisible by 3 and print "Fizz".
- If it is not divisible by 3, we check if it is divisible by 5 and print "Buzz".
- If it is

## Mistral NeMo

Compared to the other two models discussed in this lesson, Mistral NeMo is the only free model with an Apache2 License. 

It is viewed as an upgrade to the earlier open source LLM from Mistral, Mistral 7B. 

Some other feature of the NeMo model are: 

- *More efficient tokenization:* This model using the Tekken tokenizer over the more commonly used tiktoken. This allows for better performance over more languages and code. 

- *Finetuning:* The base model is available for finetuning. This allows for more flexibility for use-cases where finetuning may be needed. 

- *Native Function Calling* - Like Mistral Large, this model has been trained on function calling. This makes it unique as being one of the first open source models to do so. 


## Mistral NeMo

Compared to the other two models discussed in this lesson, Mistral NeMo is the only free model with an Apache2 License. 

It is viewed as an upgrade to the earlier open source LLM from Mistral, Mistral 7B. 

Some other feature of the NeMo model are: 

- *More efficient tokenization:* This model uses the Tekken tokenizer over the more commonly used tiktoken. This allows for better performance over more languages and code. 

- *Finetuning:* The base model is available for finetuning. This allows for more flexibility for use-cases where finetuning may be needed. 

- *Native Function Calling* - Like Mistral Large, this model has been trained on function calling. This makes it unique as being one of the first open source models to do so. 


### Comparing Tokenizers 

In this sample, we will look at how Mistral NeMo handles tokenization compared to Mistral Large. 

Both samples take the same prompt but you shoud see that NeMo returns back less tokens vs Mistral Large. 

In [4]:
# Import needed packages:
from mistral_common.protocol.instruct.messages import (
    UserMessage,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import (
    Function,
    Tool,
)
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

# Load Mistral tokenizer

model_name = "open-mistral-nemo	"

tokenizer = MistralTokenizer.from_model(model_name)

# Tokenize a list of messages
tokenized = tokenizer.encode_chat_completion(
    ChatCompletionRequest(
        tools=[
            Tool(
                function=Function(
                    name="get_current_weather",
                    description="Get the current weather",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "description": "The temperature unit to use. Infer this from the users location.",
                            },
                        },
                        "required": ["location", "format"],
                    },
                )
            )
        ],
        messages=[
            UserMessage(content="What's the weather like today in Paris"),
        ],
        model=model_name,
    )
)
tokens, text = tokenized.tokens, tokenized.text

print(len(tokens))

d:\miniconda\envs\ai\Lib\site-packages\mistral_common\tokens\tokenizers\mistral.py:176: FutureWarning: Calling `MistralTokenizer.from_model(..., strict=False)` is deprecated as it can lead to incorrect tokenizers. It is strongly recommended to use MistralTokenizer.from_model(..., strict=True)` which will become the default in `mistral_common=1.7.0`.If you are using `mistral_common` for open-sourced model weights, we recommend using `MistralTokenizer.from_file('<path/to/tokenizer/file>')` instead.
  warnings.warn(
d:\miniconda\envs\ai\Lib\site-packages\mistral_common\tokens\tokenizers\tekken.py:254: FutureWarning: Special tokens not found in d:\miniconda\envs\ai\Lib\site-packages\mistral_common\data\tekken_240718.json and default to ({'rank': 0, 'token_str': <SpecialTokens.unk: '<unk>'>, 'is_control': True}, {'rank': 1, 'token_str': <SpecialTokens.bos: '<s>'>, 'is_control': True}, {'rank': 2, 'token_str': <SpecialTokens.eos: '</s>'>, 'is_control': True}, {'rank': 3, 'token_str': <Specia

128


In [5]:
# Import needed packages:
from mistral_common.protocol.instruct.messages import (
    UserMessage,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import (
    Function,
    Tool,
)
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

# Load Mistral tokenizer

model_name = "mistral-large-latest"

tokenizer = MistralTokenizer.from_model(model_name)

# Tokenize a list of messages
tokenized = tokenizer.encode_chat_completion(
    ChatCompletionRequest(
        tools=[
            Tool(
                function=Function(
                    name="get_current_weather",
                    description="Get the current weather",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "description": "The temperature unit to use. Infer this from the users location.",
                            },
                        },
                        "required": ["location", "format"],
                    },
                )
            )
        ],
        messages=[
            UserMessage(content="What's the weather like today in Paris"),
        ],
        model=model_name,
    )
)
tokens, text = tokenized.tokens, tokenized.text

# Count the number of tokens
print(len(tokens))

135


## Learning does not stop here, continue the Journey

After completing this lesson, check out our [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) to continue leveling up your Generative AI knowledge!